# GPU runner (Kaggle / Colab)

Runs any of the repo's scripts on a free GPU and returns everything they wrote as one zip
(PLAN T44, DECISIONS D-026). Edit the **settings cell** only: the branch, the commands, the
IndicQA languages and any extra output paths.

**Kaggle:** *File → Import Notebook* → *Settings → Accelerator: GPU T4* and *Internet: on* →
*Run All*. Download the zip from the *Output* panel.
**Colab:** *File → Upload notebook* → *Runtime → Change runtime type → T4 GPU* → *Run all*.

**Gated models** (IndicNER, IndicTrans2, Llama, Gemma) need your Hugging Face token as a
secret named `HF_TOKEN`: Kaggle *Add-ons → Secrets*, Colab the key icon. The notebook reads it
from there and never prints it. Public models work without it.

The run refuses to start on a dirty clone, records the commit and GPU in every run's
`meta.json`, and pushes nothing to GitHub. Unzip the result at the repo root; the paths are
repo-relative.

In [ ]:
import os
import pathlib

# ---- settings: edit this cell only ------------------------------------------------------
BRANCH = "main"
REPO_URL = "https://github.com/YousufFFFF/indic-rag-hallu.git"
INDICQA_LANGS = ["hi"]  # downloaded with pinned, verified hashes before the commands run
COMMANDS = [  # run in order from the repo root; the first failure stops the notebook
    "python -u scripts/run_pilot_eval.py --embedder me5-small --k 10 --context-passages 3"
    " --require-clean",
]
EXTRA_OUTPUTS = ["results/m0"]  # repo paths to zip besides this session's run folders
ZIP_NAME = "gpu_results.zip"
# -----------------------------------------------------------------------------------------

WORK = pathlib.Path("/kaggle/working" if pathlib.Path("/kaggle/working").exists() else "/content")
REPO = WORK / "indic-rag-hallu"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
print("working dir:", WORK)

In [ ]:
# 1. A GPU must be attached.
import sys

import torch

print("python", sys.version.split()[0], "| torch", torch.__version__)
assert torch.cuda.is_available(), "No GPU: turn on a GPU accelerator in the notebook settings."
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# 2. Fresh clone. Step out of any earlier clone first: deleting the current directory
#    breaks every later shell command when the notebook is re-run in the same session.
%cd {WORK}
!rm -rf {REPO}
!git clone --quiet --depth 1 --branch {BRANCH} {REPO_URL} {REPO}
!git -C {REPO} log -1 --format='commit %H  %s'

In [ ]:
# 3. Dependencies at the repo's pinned versions, read from pyproject.toml so this notebook
#    can't drift from them. The platform's CUDA torch and its numpy are kept (their
#    versions are recorded in every run's meta.json); the demo-only gradio is skipped.
import subprocess
import tomllib

deps = tomllib.loads((REPO / "pyproject.toml").read_text("utf-8"))["project"]["dependencies"]
keep_platform = ("torch", "numpy", "gradio")
pins = [d for d in deps if not d.split("==")[0].strip().lower().startswith(keep_platform)]
print("installing:", " ".join(pins))
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pins], check=True)
# requires-python pins 3.11 for Docker; the code also runs on newer 3.x (D-026).
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--no-deps",
        "--ignore-requires-python",
        "-e",
        str(REPO),
    ],
    check=True,
)

In [ ]:
# 4. Hugging Face token from the platform's secrets (optional; never printed).
token = None
try:
    from kaggle_secrets import UserSecretsClient

    token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    try:
        from google.colab import userdata

        token = userdata.get("HF_TOKEN")
    except Exception:
        pass
if token:
    os.environ["HF_TOKEN"] = token
print("HF token:", "found (gated models available)" if token else "not set (public models only)")
del token

In [ ]:
# 5. IndicQA for the requested languages (pinned revision, SHA-256 verified). Checked:
#    a failed download or hash mismatch stops here, not later inside a command.
%cd {REPO}
if INDICQA_LANGS:
    dl = subprocess.run(
        [sys.executable, "scripts/download_indicqa.py", "--lang", *INDICQA_LANGS],
        cwd=REPO,
        capture_output=True,
        text=True,
    )
    print(dl.stdout + dl.stderr, end="")
    if dl.returncode != 0:
        raise RuntimeError(f"IndicQA download failed (exit {dl.returncode})")

In [ ]:
# 6. The commands, in order, from a clean tree. Output streams here as it happens.
import datetime

START = datetime.datetime.now(datetime.UTC).isoformat(timespec="seconds")
dirty = subprocess.run(
    ["git", "status", "--porcelain"], cwd=REPO, capture_output=True, text=True, check=True
).stdout.strip()
assert not dirty, f"the clone is not clean:\n{dirty}"
FAILED = None  # (command, exit code) of the first failure; later commands are skipped
for cmd in COMMANDS:
    print(f"\n$ {cmd}", flush=True)
    proc = subprocess.Popen(
        cmd, shell=True, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
    )
    for line in proc.stdout:
        print(line, end="", flush=True)
    if proc.wait() != 0:
        FAILED = (cmd, proc.returncode)
        print(f"\nFAILED (exit {proc.returncode}): {cmd}; skipping the rest, packaging anyway")
        break
if FAILED is None:
    print("\nall commands finished; session started", START)

In [ ]:
# 7. One zip: every run folder started in this session + EXTRA_OUTPUTS. Runs even after a
#    failed command (so the failed run's folder is in the zip). Any failure, of a command
#    or of packaging, is raised at the end; "Download" is printed only if a zip was written.
zip_path = WORK / ZIP_NAME
zip_path.unlink(missing_ok=True)  # an older archive must never pass for this session's
pkg_cmd = [sys.executable, "scripts/package_results.py", "--out", str(zip_path)]
pkg_cmd += ["--since", START] + [arg for p in EXTRA_OUTPUTS for arg in ("--path", p)]
if FAILED:
    pkg_cmd.append("--allow-missing")  # a crash may not have written every output
pkg = subprocess.run(pkg_cmd, cwd=REPO, capture_output=True, text=True)
print(pkg.stdout + pkg.stderr, end="")
problems = []
if FAILED:
    problems.append(f"command failed (exit {FAILED[1]}): {FAILED[0]}")
if pkg.returncode != 0 or not zip_path.exists():
    problems.append(f"packaging failed (exit {pkg.returncode}); no zip was written")
else:
    print("Download", ZIP_NAME, "from the Output panel / file browser, then stop the session.")
if problems:
    raise RuntimeError("; ".join(problems))